In [ ]:
import subscriptionManager as smSL
import feedbackManager as fmSL

# Checks if game is video game 

def search_Vid(game_ID):
    v = open('Video_Game_Info.txt', 'r')
    for line in v: 
        ID = line.split(",")[0]
        if game_ID == ID:
            return True 
    v.close()
    return False 
    

# Checks if game is board game 

def search_Board(game_ID):
    b = open('Board_Game_Info.txt', 'r')
    for line in b:
        ID = line.split(",")[0]
        if game_ID == ID:
            return True 
    b.close()
    return False 
    

# Prints all video games 

def vid_Games():
    v = open('Video_Game_Info.txt', 'r')
    for line in v.readlines()[1:]:
        name = line.split(",")[1]
        print(name)
    v.close()

# Prints all board games 

def board_Games():
    b = open('Board_Game_Info.txt', 'r')
    for line in b.readlines()[1:]:
        name = line.split(",")[1]
        print(name)
    b.close()

# Prints all video games of a specific genre 

def vid_genre(search_genre): 
    v = open('Video_Game_Info.txt', 'r')
    for line in v.readlines()[1:]:
        name = line.split(",")[1]
        genre = line.split(",")[3]
        if genre == search_genre: 
            print(name)
    v.close()

# Prints all board games of a specific genre 

def board_genre(search_genre): 
    b = open('Board_Game_Info.txt', 'r')
    for line in b.readlines()[1:]:
        name = line.split(",")[1]
        genre = line.split(",")[3]
        if genre == search_genre: 
            print(name)
    b.close()

# Returns a filtered list of games based on search criteria

def filter_games(genre, entry):

    r = open("Rental.txt", "r")
    unavaliable = []
    for line in r.readlines()[1:]:
        if line.split(",")[2] == "N/A":
            unavaliable.append(line)
    r.close()

    if genre == "board":
        r = open("Board_Game_Info.txt", "r")
        board_games = r.readlines()[1:]
        board_game_dict = {}
        for line in board_games:
            line = line.split(",")
            del line[4]
            board_game_dict[line[1]] = line

        fit_board_criteria = []
        fit_board_criteria.append('GameID, Name, No.Players, Genre')
        for value in board_game_dict.values():
            for i in value: 
                if i == entry:
                    fit_board_criteria.append(', '.join(i for i in board_game_dict[value[1]]))

        for entry in unavaliable:
            for line in fit_board_criteria:
                if entry.split(",")[0] == line.split(", ")[0]:
                    fit_board_criteria.remove(line)

        r.close()
        return fit_board_criteria
    
    
    elif genre == "video":
        r = open("Video_Game_Info.txt", "r") 
        video_games= r.readlines()[1:]
        video_game_dict = {}
        for line in video_games:
            line = line.split(",")
            del line[4]
            video_game_dict[line[1]] = line
        
        fit_video_criteria = []
        fit_video_criteria.append('GameID, Name, Platform, Genre')
        for value in video_game_dict.values():
            for i in value: 
                if i == entry:
                    fit_video_criteria.append(', '.join(i for i in video_game_dict[value[1]]))
        
        for entry in unavaliable:
            for line in fit_video_criteria:
                if entry.split(",")[0] == line.split(", ")[0]:
                    fit_video_criteria.remove(line)

        r.close()
        return fit_video_criteria

    else:
        return 


# Allows store manager to all Video games that are avaliable for rent

def avaliable_vid_games(): 

    r = open("Rental.txt", "r")

    unavaliable = []
    avaliable = []

    for line in r.readlines()[1:]:
        if line.split(",")[2] == "N/A":
            unavaliable.append(line)

    r.close()

    r = open("Video_Game_Info.txt", "r")
    avaliable.append('GameID, Name, Platform, Genre')

    for line in r.readlines()[1:]:
        line.replace('\n', '')
        line = line.split(",")
        line.pop(4)
        line = ', '.join(line)
        avaliable.append(line)

    for entry in unavaliable:
        for line in avaliable:
            if entry.split(",")[0] == line.split(", ")[0]:
                avaliable.remove(line)
    
    return avaliable

# Allows store manager to see all Board games that are avaliable for rent 

def avaliable_brd_games(): 

    r = open("Rental.txt", "r")

    unavaliable = []
    avaliable = []

    for line in r.readlines()[1:]:
        if line.split(",")[2] == "N/A":
            unavaliable.append(line)

    r.close()

    r = open("Board_Game_Info.txt", "r")
    avaliable.append("GameID, Name, No.Players, Genre")

    for line in r.readlines()[1:]:
        line.replace('\n', '')
        line = line.split(",")
        line.pop(4)
        line = ', '.join(line)
        avaliable.append(line)

    for entry in unavaliable:
        for line in avaliable:
            if entry.split(",")[0] == line.split(", ")[0]:
                avaliable.remove(line)
      
    return avaliable



In [ ]:

from database import is_subscribed 
from database import rental_lim
from datetime import datetime, timedelta

rent_status = None

# Allows user to rent game (if avaliable) and adds it to Rental.txt

def game_Rent(user_ID, game_ID):
    
    global rent_status

    if is_subscribed(user_ID):

        if search_Vid(game_ID) or search_Board(game_ID):

            rent_db = open('Rental.txt', 'r')
            rental_Date = datetime.now().strftime("%Y-%m-%d")

            rent_lim = rental_lim(user_ID)
            year = int(rental_Date.replace('-', ' ').split()[0])
            month = int(rental_Date.replace('-', ' ').split()[1])
            day = int(rental_Date.replace('-', ' ').split()[2])

            latest_return_Date = datetime(year, month, day) + timedelta(weeks=rent_lim)
            latest_return_Date = str(latest_return_Date).replace(" 00:00:00", "")

            for line in rent_db.readlines()[1:]:
                id = line.split(',')[0]
                returned = line.split(',')[2]
                if game_ID == id and returned == 'N/A':
                    rent_status = "already rented"
                    return
        
            else: 
                rent_db.close()
                write_db = open('Rental.txt', 'a')
                entry = f"\n{game_ID},{rental_Date},N/A,{user_ID},{latest_return_Date}"
                write_db.write(entry)
                rent_status = "success"
                return
            
        else: 
            rent_status = "not exist"
            return

    else: 
       rent_status = "not subscribed"
       return


In [ ]:
import subscriptionManager as smSL
import feedbackManager as fmSL
from datetime import datetime


return_status = None


def game_Return(game_ID, user_ID):

    global return_status

    rental_logs = []

    r = open("Rental.txt", "r")

    for line in r.readlines()[1:]:

        rental_logs.append(line.replace("\n", ""))

    return_history = []

    for line in rental_logs:
        if line.split(",")[0] == game_ID:
            return_history.append(line.split(",")[2])
    
    r.close()

    if "N/A" in return_history:

        w = open("Rental.txt", "w")
        w.write("GameID,RentalDate,ReturnDate,RentalCustomerID,LatestReturnDate")

        for line in rental_logs: 

            game_id = line.split(",")[0]
            rental_date = line.split(",")[1]
            return_date = line.split(",")[2]
            user_id = line.split(",")[3]
            latest_date = line.split(",")[4]
            current_date = datetime.now().strftime("%Y-%m-%d")
            concat_latest_date = int(latest_date.replace("-", ""))
            concat_return_date = int(current_date.replace("-", ""))

            if all([game_id == game_ID, user_id == user_ID, return_date == "N/A"]):

                w.write(f"\n{game_id},{rental_date},{current_date},{user_id},{latest_date}")

                if concat_return_date > concat_latest_date: 
                    return_status = "Late"

                elif concat_return_date < concat_latest_date: 
                    return_status = "On Time"            

            else: 
                w.write(f"\n{line}")
    
        w.close()
    
    else:
        return_status = "Invalid"



def submit_review(game_ID, rating, comment):

    a = open("game_feedback.txt", "a")
    new_review = f'{game_ID},{rating},{comment}'
    a.write("\n" + new_review)
    a.close()


In [ ]:
import subscriptionManager as smSL
import datetime
from database import is_subscribed
from datetime import datetime, timedelta

# Function checking avaliability of day and time slot 

def avaliability(date, time_slot): 

  

    r = open("Booking.txt", "r")
    selected_date = date.strftime("%Y-%m-%d")

    slot_data = []
    for line in r.readlines()[1:]:
        if line[1] == selected_date and line[2] == time_slot:
            slot_data.append(int(line[3]) + 1)
        
    num_ppl = sum(slot_data)

    return (num_ppl, 50 - num_ppl)
 
    r.close()
        

    
# Function adding new entry to Booking.txt 

def booking(user_ID, date, time_slot, No_Guests):

    r = open("Booking.txt", "r")
    logs = r.readlines()[1:]
    r.close()

    for line in logs:
        if line.split(",")[0] == user_ID and line.split(",")[1] == date and line.split(",")[2] == time_slot:
            return "invalid"
    
    a = open("Booking.txt", "a")
    a.write(f"\n{user_ID},{date},{time_slot},{No_Guests}")
    return "success"
    


# Function which checks if inputted date is within 1 week of current date:

def within_a_week(date):

    selected_date = date.strftime("%Y-%m-%d")
    current_date = datetime.now().strftime("%Y-%m-%d")

    year = int(current_date.replace('-', ' ').split()[0])
    month = int(current_date.replace('-', ' ').split()[1])
    day = int(current_date.replace('-', ' ').split()[2])

    week_time = datetime(year, month, day) + timedelta(weeks=1)
    week_time = str(week_time).replace(" 00:00:00", "")

    concat_week_time = int(week_time.replace("-", ""))
    concat_selected_date = int(selected_date.replace("-", ""))
    concat_current_date = int(current_date.replace("-", ""))

    if concat_week_time < concat_selected_date: 
        return False 
    
    elif concat_selected_date < concat_current_date: 
        return False
    
    else:
        return True

In [ ]:
import subscriptionManager as smSL
import feedbackManager as fmSL
import matplotlib.pyplot as plt

feedback_list = fmSL.load_feedback()


# Returns a dictionary of games and their rental frequency

def all_rentings():

    r = open("Rental.txt", "r")

    freq_dict = {}
    
    for line in r.readlines()[1:]:
        game_id = line.split(",")[0]
        
        if freq_dict.get(game_id) == True:
            freq_dict[game_id] += 1

        elif freq_dict.get(game_id) == None:
            freq_dict[game_id] = 1 
    
    r.close()

    r = open("Video_Game_Info.txt")

    for line in r.readlines()[1:]:
        game_id = line.split(",")[0]

        if freq_dict.get(game_id) == None:
            freq_dict[game_id] = 0
    
    r.close()

    r = open("Board_Game_Info.txt")

    for line in r.readlines()[1:]:
        game_id = line.split(",")[0]

        if freq_dict.get(game_id) == None:
            freq_dict[game_id] = 0
    
    r.close()

    return(freq_dict)



# Returns a dictionary of games and their mean rating 

def average_ratings():
    rated_games = []

    for game in feedback_list:
        if game['GameID'] in rated_games:
            pass 
        else:
            rated_games.append((game['GameID'], []))


    for game in rated_games:
        for review in feedback_list:
            if review['GameID'] == game[0]:
                game[1].append(review['Rating'])

    mean_ratings = {}           
    
    for game in rated_games:

        summed = 0
        for rating in game[1]:
            summed += rating 

        mean = summed / len(game[1])

        mean_ratings[game[0]] = mean
    
    return mean_ratings



# Returns list of IDs of least rented games

def least_rented():

    freq_dict = all_rentings()

    unpopular = []
    for game in freq_dict.items():
        if game[1] == min(freq_dict.values()):
            unpopular.append(game[0])

    return unpopular



# Returns game with lowest Rating 

def worst_rating():

    mean_ratings = average_ratings()

    worst_rated = []

    for game in mean_ratings.items():
        if game[1] == min(mean_ratings.values()):
            worst_rated.append(game)
    
    return worst_rated
    


# Function to find name of a game from its ID

def id_to_name(id):
    
    v = open("Video_Game_Info.txt", "r")
    b = open("Board_Game_Info.txt", "r")

    game_names = []

    for line in v.readlines()[1:]:
        game_id = line.split(",")[0]
        game_name = line.split(",")[1]
        game_names.append((game_id, game_name))
    
    for line in b.readlines()[1:]:
        game_id = line.split(",")[0]
        game_name = line.split(",")[1]
        game_names.append((game_id, game_name))
    
    v.close()
    b.close()

    for entry in game_names:
        if entry[0] == id:
            return entry[1]
        

# Defining Bar Chart analytics

def plot(type):

    with analytics_output:

        if type == "frequency":

            plt.close('all')
            x = all_rentings().keys() 
            y = all_rentings().values()
            plt.figure(figsize=(5, 3))
            plt.bar(x, y)
            plt.xlabel("Game ID")
            plt.ylabel("Rental Freq")
            plt.ylim(0, 10)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
    
        if type == "rating":

            plt.close('all')
            x = average_ratings().keys()
            y = average_ratings().values()
            plt.figure(figsize=(5, 3))
            plt.bar(x, y)
            plt.xlabel("Game ID")
            plt.ylabel("Mean Rating")
            plt.ylim(0, 5)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

In [ ]:
# Cell defining all the widgets that appear in the GUI


import ipywidgets as widgets 
from ipywidgets import Layout 
from ipywidgets import Box
from ipywidgets import VBox


rent_button = widgets.Button(
    value=False,
    description='Rent Games',
    disabled=False,
)

return_button = widgets.Button(
    value=False,
    description='Return Games',
    disabled=False
)

sessions_button = widgets.Button(
    value=False,
    description='Book Sessions',
    disabled=False,
)

pruning_button = widgets.Button(
    value=False,
    description='Prune Inventory',
    disabled=False
)

exit_button = widgets.Button(
    value=False,
    description='Exit',
    disabled=False
)

game_ID_text = widgets.Text(
    placeholder="Enter Game ID",
    disabled=False
)

user_ID_text = widgets.Text(
    placeholder="Enter User ID",
    disabled=False
)

user_ID_text.continuous_update = False

game_rent_submit_button = widgets.Button(
    value=False,
    description='Submit',
    disabled=False
)

view_games_button = widgets.Button(
    value=False, 
    description='Avaliable Games',
    disabled=False
)

board_games_button = widgets.Button(
    value=False, 
    description='Board Games',
    disabled=False
)

filter_board_games = widgets.Text(
    placeholder="Filter",
    disabled=False,
    layout=Layout(width='180px')
)

filter_board_search = widgets.Button(
    value=False,
    description='Search',
    disabled=False,
    layout=Layout(width='70px')
)

video_games_button = widgets.Button(
    value=False, 
    description='Video Games',
    disabled=False
)

filter_video_games = widgets.Text(
    placeholder="Filter",
    disabled=False,
    layout=Layout(width='180px')
)

filter_video_search = widgets.Button(
    value=False,
    description="Search",
    disabled=False,
    layout=Layout(width="70px")
)

game_return_submit_button = widgets.Button(
    value=False,
    description='Submit',
    disabled=False
)

enter_review_text = widgets.Text(
    disabled=False
)

enter_rating_dropdown = widgets.Dropdown(
    options=[('1 Star', 1), ('2 Star', 2), ('3 Star', 3), ('4 Star', 4), ('5 Star', 5)],
    value=1,
    layout=Layout(width="150px"),
    disabled=False
)

yes_review = widgets.Button(
    value=False, 
    description='Yes',
    disabled=False
)

no_review = widgets.Button(
    value=False, 
    description='no',
    disabled=False
)

review_submit = widgets.Button(
    value=False, 
    description="Submit",
    disabled=False
)

sessions_date = widgets.DatePicker(
    disabled=False,
    layout=Layout(width="180px")
)

slot_toggle = widgets.ToggleButtons(options=[('2-6 pm', '2pm'), ('6-10 pm', '6pm')], 
                                    disabled=False, 
                                    layout=Layout(display="flex", flex_flow="row wrap"), 
                                    style={'button_width': '100px'})

book_slot = widgets.Button(
    value=False,
    description="Book", 
    disabled=False
)

guests_dropdown = widgets.Dropdown(options=['0', '1', '2', '3'], 
                                   value='0', 
                                   description="No. Guests", 
                                   layout=Layout(width='200px'), 
                                   disabled=False)

toggle_analytics = widgets.ToggleButtons(options=['Rental Freq', 'Mean Ratings'],
                                         style={'button_width': '125px'})


on_time = widgets.HTML(value="Game Returned on Time!")

leave_review = widgets.HTML(value="Would you like to leave a review?")

already_rented = widgets.HTML(value="Failed to Rent! Game is already Rented!")

successful_rent = widgets.HTML(value="Game successfully Rented!")

game_not_exist = widgets.HTML(value="Failed to Rent! This game does not exist!")

not_subscribed = widgets.HTML(value="Failed to Rent! User is not Subscribed!")

returned_late = widgets.HTML(value="Game was returned Late!")

not_to_return = widgets.HTML(value="Game is not out for Return!")

comment_and_rate = widgets.HTML(value="Leave a comment & rate the game:")

review_and_rating_submitted = widgets.HTML(value="Your review has been submitted!")

select_a_date = widgets.HTML(value="Select a Date:", layout=Layout(display="flex", padding="0px 50px"))

week_in_advance = widgets.HTML(value="Sessions are only bookable a week in advance")

select_time = widgets.HTML(value="Select a Time Slot:", layout=Layout(display="flex", padding="0px 50px"))

enter_usr_id = widgets.HTML(value="Enter User ID:", layout=Layout(display="flex", padding="0px 75px"))

invalid_usr_id = widgets.HTML(value="Invalid User ID", layout=Layout(display="flex", padding="0px 75px"))

already_booked = widgets.HTML(valuue="User has already booked this slot!")

booking_success = widgets.HTML(value="Slot successfully booked!")

display_analytics = widgets.HTML(value="Toggle an Analytic to display", layout=Layout(padding='0px 100px'))





In [ ]:
# Cell defining relevant output widgets:


import ipywidgets as widgets 
from ipywidgets import Layout 
from ipywidgets import Box
from ipywidgets import VBox

rent_output = widgets.Output()
return_output = widgets.Output()
pruning_output = widgets.Output()
sessions_output = widgets.Output()
avaliable_games_output = widgets.Output()
board_games_output = widgets.Output()
analytics_output = widgets.Output()

In [ ]:
# Cell defining the primary boxes of the GUI


import ipywidgets as widgets 
from ipywidgets import Layout 
from ipywidgets import Box
from ipywidgets import VBox


# Box containing the 5 initial buttons 

button_box = Box(children=[view_games_button, rent_button, return_button, sessions_button, pruning_button],
                 layout=Layout(display='flex',
                               flex_flow='row wrap',
                               justify_content='center',
                               align_items='center',
                               align_content='center',
                               width = '200px'))

# Box containing the output widgets for each initial button

output_box = Box(children=[avaliable_games_output, rent_output, return_output, sessions_output, pruning_output],
                 layout=Layout(display='flex',
                               flex_flow='column wrap',
                               align_items='center',
                               align_content='center'))

# Box containing the output widgets for the initial Rent button

rent_box = Box(children=[game_ID_text, user_ID_text, game_rent_submit_button, exit_button],
               layout=Layout(display='flex',
                             flex_flow='row wrap',
                             justify_content = 'center',
                             align_items='center',
                             align_content='center',
                             width='200px'))

# Box containing the output widgets for the initial Return button 

return_box = Box(children=[game_ID_text, user_ID_text, game_return_submit_button, exit_button],
                 layout=Layout(display='flex',
                               flex_flow='row wrap',
                               justify_content='center',
                               align_items='center',
                               align_content='center',
                               width='200px'))

# Box containing the output widgets for the initial View Games Button 

view_games_box = Box(children=[board_games_button, video_games_button, exit_button],
                     layout=Layout(display='flex',
                                   flex_flow='row wrap',
                                   justify_content='center',
                                   align_items='center',
                                   align_content='center',
                                   width='200px'))

# Box containing widgets to review games upon return 

review_box = Box(children=[comment_and_rate, enter_review_text, enter_rating_dropdown, review_submit, exit_button],
                 layout=Layout(display='flex', 
                               flex_flow='row wrap',
                               justify_content='center',
                               align_items='center',
                               align_content='center',
                               width='300px'))

# Box containing widgets for the initial Booking button 

booking_box = Box(children=[select_a_date, sessions_date, exit_button],
                  layout=Layout(display='flex', 
                               flex_flow='row wrap',
                               justify_content='center',
                               align_items='center',
                               align_content='center',
                               width='300px'))

# Box which contains the entire GUI 

universe_box = Box(children=[button_box, output_box],
                   layout=Layout(display='flex',
                                 flex_flow='row wrap',
                                justify_content='center',
                                align_items='center',
                                align_content='center',
                                width='100%',
                                height='100%'))

In [ ]:
# Cell defining the functionality of the widgets 


import ipywidgets as widgets 
from ipywidgets import Layout 
from ipywidgets import Box
from ipywidgets import VBox


def on_rent_button_clicked(b):

    rent_box.layout.display = 'flex'
    button_box.layout.display = 'none'
    rent_output.layout.display = 'flex'
    rent_output.layout.justify_content='center'
    rent_output.layout.align_items='center'
    rent_output.layout.align_content='center'
    with rent_output:
        display(rent_box)


rent_button.on_click(on_rent_button_clicked)


def on_return_button_clicked(b):

    return_box.layout.display = 'flex'
    button_box.layout.display = 'none'
    return_output.layout.display = 'flex'
    return_output.layout.justify_content='center'
    return_output.layout.align_items='center'
    return_output.layout.align_content='center'
    with return_output:
        display(return_box)


return_button.on_click(on_return_button_clicked)


def on_pruning_button_clicked(b):

    button_box.layout.display = 'none'
    pruning_output.layout.display = 'flex'
    
    with pruning_output:

        least_rented_games = []
        for game in least_rented():
            least_rented_games.append(id_to_name(game))

        popularities = widgets.HTML(value="These are the least rented games:")
        popularity_dropdown = widgets.Dropdown(options=least_rented_games, layout=Layout(width='100px'))
        popularities_box = widgets.Box(children=[popularities, popularity_dropdown])
        display(widgets.Box(children=[popularities_box, display_analytics, toggle_analytics, analytics_output, exit_button],
                            layout=Layout(display='flex',
                            flex_flow='row wrap',
                            justify_content='center',
                            align_items='center',
                            align_content='center',
                            width='500px')))
    
    plot('frequency')
        


pruning_button.on_click(on_pruning_button_clicked)


def on_toggle_analytics_change(change):
    
    if change['new'] == 'Rental Freq':
        
        pruning_output.clear_output()
        analytics_output.clear_output()

        with pruning_output:

            least_rented_games = []
            for game in least_rented():
                least_rented_games.append(id_to_name(game))

            popularities = widgets.HTML(value="These are the least rented games:")
            popularity_dropdown = widgets.Dropdown(options=least_rented_games, layout=Layout(width='100px'))
            popularities_box = widgets.Box(children=[popularities, popularity_dropdown])
            display(widgets.Box(children=[popularities_box, display_analytics, toggle_analytics, analytics_output, exit_button],
                            layout=Layout(display='flex',
                            flex_flow='row wrap',
                            justify_content='center',
                            align_items='center',
                            align_content='center',
                            width='500px')))
    
        plot('frequency')

    if change['new'] == 'Mean Ratings':

        pruning_output.clear_output()
        analytics_output.clear_output()
        
        with pruning_output:

            least_rented_games = []
            for game in least_rented():
                least_rented_games.append(id_to_name(game))

            popularities = widgets.HTML(value="These are the least rented games:")
            popularity_dropdown = widgets.Dropdown(options=least_rented_games, layout=Layout(width='100px'))
            popularities_box = widgets.Box(children=[popularities, popularity_dropdown])
            display(widgets.Box(children=[popularities_box, display_analytics, toggle_analytics, analytics_output, exit_button],
                            layout=Layout(display='flex',
                            flex_flow='row wrap',
                            justify_content='center',
                            align_items='center',
                            align_content='center',
                            width='500px')))
    
        plot('rating')

toggle_analytics.observe(on_toggle_analytics_change, names='value')




def on_avaliable_games_clicked(b):

    video_games_button.layout.display = 'block'
    board_games_button.layout.display = 'block'
    button_box.layout.display = 'none'
    avaliable_games_output.layout.display = 'flex'
    with avaliable_games_output: 
        display(view_games_box) 


view_games_button.on_click(on_avaliable_games_clicked)


def on_board_games_button_clicked(b):

    avaliable_games_output.clear_output()
    
    with avaliable_games_output: 

        filter_board_box = Box(children=[filter_board_games, filter_board_search],
                               layout=Layout(width="250px"))
        
        display(filter_board_box)
        
        avaliable_board_games = VBox(children=[widgets.Label(entry) for entry in avaliable_brd_games()],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='flex-start',
                                    align_items='center',
                                    align_content='center',
                                    width='250px'))
        
        display(avaliable_board_games)
        display(exit_button)


board_games_button.on_click(on_board_games_button_clicked)


def on_filter_board_search_clicked(b):

    avaliable_games_output.clear_output()

    with avaliable_games_output:

        filter_board_box = Box(children=[filter_board_games, filter_board_search],
                               layout=Layout(width="250px"))
        
        display(filter_board_box)

        if filter_board_games.value == "": 
            avaliable_board_games = VBox(children=[widgets.Label(entry) for entry in avaliable_brd_games()],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='flex-start',
                                    align_items='center',
                                    align_content='center',
                                    width='250px'))
        
            display(avaliable_board_games)
        
        else: 
            filtered_board_games = VBox(children=[widgets.Label(entry) for entry in filter_games("board", filter_board_games.value)],
                                        layout=Layout(display='flex',
                                        flex_flow='row wrap',
                                        justify_content='flex-start',
                                        align_items='center',
                                        align_content='center',
                                        width='250px'))
        
            display(filtered_board_games)

        display(exit_button)


filter_board_search.on_click(on_filter_board_search_clicked)



def on_video_games_button_clicked(b):

    avaliable_games_output.clear_output()

    with avaliable_games_output: 

        filter_video_box = Box(children=[filter_video_games, filter_video_search],
                               layout=Layout(width="250px"))
        
        display(filter_video_box)

        avaliable_video_games = VBox(children=[widgets.Label(entry) for entry in avaliable_vid_games()],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='flex-start',
                                    align_items='center',
                                    align_content='center',
                                    width='250px'))
        display(avaliable_video_games)
        display(exit_button)


video_games_button.on_click(on_video_games_button_clicked)


def on_filter_video_search_clicked(b):

    avaliable_games_output.clear_output()

    with avaliable_games_output:

        filter_video_box = Box(children=[filter_video_games, filter_video_search],
                               layout=Layout(width="250px"))
        
        display(filter_video_box)

        if filter_video_games.value == "": 
            avaliable_video_games = VBox(children=[widgets.Label(entry) for entry in avaliable_vid_games()],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='flex-start',
                                    align_items='center',
                                    align_content='center',
                                    width='250px'))
        
            display(avaliable_video_games)
        
        else: 
            filtered_video_games = VBox(children=[widgets.Label(entry) for entry in filter_games("video", filter_video_games.value)],
                                        layout=Layout(display='flex',
                                        flex_flow='row wrap',
                                        justify_content='flex-start',
                                        align_items='center',
                                        align_content='center',
                                        width='250px'))
        
            display(filtered_video_games)

        display(exit_button)


filter_video_search.on_click(on_filter_video_search_clicked)


def on_game_rent_submit_button_clicked(b):

    game_ID = game_ID_text.value 
    user_ID = user_ID_text.value
    rent_output.clear_output()

    with rent_output: 

        game_Rent(user_ID, game_ID)

        if rent_status == "already rented":
            display(widgets.VBox(children=[already_rented, exit_button],
                                 layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        elif rent_status == "success":
            display(widgets.VBox(children=[successful_rent, exit_button],
                                 layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        elif rent_status == "not exist": 
            display(widgets.VBox(children=[game_not_exist, exit_button],
                                 layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        elif rent_status == "not subscribed":
            display(widgets.VBox(children=[not_subscribed, exit_button],
                                 layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))


game_rent_submit_button.on_click(on_game_rent_submit_button_clicked)


def on_game_return_submit_button_clicked(b):

    game_ID = game_ID_text.value 
    user_ID = user_ID_text.value 
    return_output.clear_output()

    with return_output:

        game_Return(game_ID, user_ID)

        if return_status == "On Time":
            display(widgets.VBox(children=[on_time, leave_review, yes_review, no_review],
                                 layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
 

        elif return_status == "Late": 
            display(widgets.VBox(children=[returned_late, exit_button],
                                 layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="200px")))

        elif return_status == "Invalid":
            display(widgets.VBox(children=[not_to_return, exit_button],
                                 layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))


game_return_submit_button.on_click(on_game_return_submit_button_clicked)


def on_yes_review_clicked(b):
    
    return_output.clear_output() 

    with return_output: 
        display(review_box)


yes_review.on_click(on_yes_review_clicked)


def on_no_review_clicked(b):

    return_output.clear_output()
    return_output.layout.display='none'
    button_box.layout.display='flex'
    game_ID_text.value = ""
    user_ID_text.value = ""


no_review.on_click(on_no_review_clicked)


def on_review_submit_clicked(b): 
    
    submit_review(game_ID_text.value, enter_rating_dropdown.value, enter_review_text.value)
    return_output.clear_output()
    game_ID_text.value = ""
    user_ID_text.value = ""

    with return_output: 
        review_submitted = widgets.Box(children=[review_and_rating_submitted, exit_button],
                                       layout=Layout(display='flex',
                                        flex_flow='row wrap',
                                        justify_content='center',
                                        align_items='center',
                                        align_content='center',
                                        width="250px"))
        display(review_submitted)


review_submit.on_click(on_review_submit_clicked)


def on_sessions_button_clicked(b):

    sessions_output.clear_output()
    button_box.layout.display = 'none'
    sessions_output.layout.display = 'flex'

    with sessions_output:
        display(booking_box)


sessions_button.on_click(on_sessions_button_clicked)


def on_sessions_date_changed(d):

    user_ID_text.value = ""
    date = sessions_date.value

    sessions_output.clear_output()
    
    if within_a_week(date) == False:
        
        with sessions_output:

            display(widgets.Box(children=[select_a_date, sessions_date, week_in_advance, exit_button],
                                       layout=Layout(display='flex',
                                        flex_flow='row wrap',
                                        justify_content='center',
                                        align_items='center',
                                        align_content='center',
                                        width="350px")))
    
    elif within_a_week(date) == True:
        
        with sessions_output:

            user_ID_text.layout.width = "150px"

            display(widgets.Box(children=[select_a_date, sessions_date, enter_usr_id, user_ID_text, exit_button],
                                layout=Layout(display='flex',
                                flex_flow='row wrap',
                                justify_content='center',
                                align_items='center',
                                align_content='center',
                                width="300px")))
        
    
sessions_date.observe(on_sessions_date_changed, names='value')


def on_enter_usr_id_submit(b):


    if is_subscribed(user_ID_text.value) == True: 

        num_ppl = avaliability(sessions_date.value, '2pm')[0]
        avaliable = avaliability(sessions_date.value, '2pm')[1]

        slots_remaining = widgets.HTML(value=f"{num_ppl}/50 spaces booked: {avaliable} spaces remaining")

        sessions_output.clear_output()

        if avaliable == 0:

            fully_booked = widgets.HTML(value="This slot is fully booked!")

            with sessions_output: 

                display(widgets.Box(children=[select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, fully_booked, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        
        elif avaliable == 1:

            display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))

        elif avaliable == 2:

            guests_dropdown.options = ['0', '1']

            with sessions_output:

                display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, guests_dropdown, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        
        elif avaliable == 3: 
            
            guests_dropdown.options = ['0', '1', '2']

            with sessions_output:

                display(widgets.Box(children=[select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, guests_dropdown, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        
        elif avaliable > 3: 
        
            guests_dropdown.options = ['0', '1', '2', '3']

            with sessions_output:

                display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, guests_dropdown, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
    
    elif is_subscribed(user_ID_text.value) == False: 

        sessions_output.clear_output()

        with sessions_output:
        
            display(widgets.Box(children=[select_a_date, sessions_date, enter_usr_id, user_ID_text, invalid_usr_id, exit_button],
                                layout=Layout(display='flex',
                                flex_flow='row wrap',
                                justify_content='center',
                                align_items='center',
                                align_content='center',
                                width="300px")))


user_ID_text.observe(on_enter_usr_id_submit, 'value')


def on_slot_toggle_change(change):

    if change['new'] == '2-6 pm':

        num_ppl = avaliability(sessions_date.value, '2pm')[0]
        avaliable = avaliability(sessions_date.value, '2pm')[1]

        slots_remaining = widgets.HTML(value=f"{num_ppl}/50 spaces booked: {avaliable} spaces remaining")

        sessions_output.clear_output()

        if avaliable == 0:

            fully_booked = widgets.HTML(value="This slot is fully booked!")

            with sessions_output: 

                display(widgets.Box(children=[select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, fully_booked, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        
        elif avaliable == 1:

            display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))

        elif avaliable == 2:

            guests_dropdown.options = ['0', '1']
            with sessions_output:

                display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, guests_dropdown, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        
        elif avaliable == 3: 
            
            guests_dropdown.options = ['0', '1', '2']

            with sessions_output:

                display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, guests_dropdown, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        
        elif avaliable > 3: 
        
            guests_dropdown.options = ['0', '1', '2', '3']
            with sessions_output:

                display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, guests_dropdown, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))


    elif change['new'] == '6-10 pm':
        
        num_ppl = avaliability(sessions_date.value, '6pm')[0]
        avaliable = avaliability(sessions_date.value, '6pm')[1]

        slots_remaining = widgets.HTML(value=f"{num_ppl}/50 spaces booked: {avaliable} spaces remaining")

        sessions_output.clear_output()

        if avaliable == 0:

            fully_booked = widgets.HTML(value="This slot is fully booked!")

            with sessions_output: 

                display(widgets.Box(children=[select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, fully_booked, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        
        elif avaliable == 1:

            display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))

        elif avaliable == 2:

            guests_dropdown.options = ['0', '1']

            with sessions_output:

                display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, guests_dropdown, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        
        elif avaliable == 3: 
            
            guests_dropdown.options = ['0', '1', '2']
            with sessions_output:

                display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, guests_dropdown, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))
        
        elif avaliable > 3: 
        
            guests_dropdown.options = ['0', '1', '2', '3'] 

            with sessions_output:

                display(widgets.Box([select_a_date, sessions_date, select_time, slot_toggle, slots_remaining, guests_dropdown, book_slot, exit_button],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='center',
                                    align_items='center',
                                    align_content='center',
                                    width="300px")))


slot_toggle.observe(on_slot_toggle_change, names='value')


def on_book_slot_clicked(b):

    user_id = user_ID_text.value
    selected_date = sessions_date.value.strftime("%Y-%m-%d")
    selected_slot = slot_toggle.value 
    selected_guests = guests_dropdown.value

    sessions_output.clear_output()

    with sessions_output:

        if booking(user_id, selected_date, selected_slot, selected_guests) == "invalid":
            display(already_booked)
            display(exit_button)
        
        else:

            display(booking_success)
            display(exit_button)


book_slot.on_click(on_book_slot_clicked)


def on_exit_button_clicked(b):

    sessions_output.clear_output()
    pruning_output.clear_output()
    return_output.clear_output()
    rent_output.clear_output()
    avaliable_games_output.clear_output()
    analytics_output.clear_output()

    game_ID_text.value = ""
    user_ID_text.value = ""
    filter_video_games.value = ""
    filter_board_games.value = ""

    sessions_output.layout.display = 'none'
    pruning_output.layout.display = 'none'
    return_output.layout.display = 'none'
    rent_output.layout.display = 'none'
    avaliable_games_output.display = 'none'
    button_box.layout.display = 'flex'


exit_button.on_click(on_exit_button_clicked)


# Initial state of the GUI

display(universe_box)
